In [ ]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

In [1]:
import torch
print("PyTorch:", torch.__version__)

import transformers
print("Transformers:", transformers.__version__)

print("Step 1 and Step 2 successful")

PyTorch: 2.14.0+cpu
Transformers: 5.17.0
Step 1 and Step 2 successful


In [2]:
import torch

print("PyTorch:", torch.__version__)

import transformers

print("Transformers:", transformers.__version__)

import datasets

print("Datasets:", datasets.__version__)

print("All imports successful!")

PyTorch: 2.14.0+cpu
Transformers: 5.17.0
Datasets: 5.0.1
All imports successful!


In [3]:
import os
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

PyTorch version: 2.14.0+cpu
CUDA available: False


In [4]:
train_path = "../data/train.csv"
test_path = "../data/test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nNumber of categories:", train_df["category"].nunique())

Train shape: (10003, 2)
Test shape: (3080, 2)

Columns:
['text', 'category']

Number of categories: 77


In [5]:
print("Missing values in train:")
print(train_df.isnull().sum())

print("\nMissing values in test:")
print(test_df.isnull().sum())

print("\nDuplicate rows in train:", train_df.duplicated().sum())
print("Duplicate rows in test:", test_df.duplicated().sum())

Missing values in train:
text        0
category    0
dtype: int64

Missing values in test:
text        0
category    0
dtype: int64

Duplicate rows in train: 0
Duplicate rows in test: 0


In [6]:
categories = sorted(train_df["category"].unique())

label2id = {label: idx for idx, label in enumerate(categories)}
id2label = {idx: label for label, idx in label2id.items()}

train_df["label"] = train_df["category"].map(label2id)
test_df["label"] = test_df["category"].map(label2id)

print("Number of labels:", len(label2id))

print("\nFirst 10 mappings:")
for i in range(10):
    print(i, "→", id2label[i])

Number of labels: 77

First 10 mappings:
0 → Refund_not_showing_up
1 → activate_my_card
2 → age_limit
3 → apple_pay_or_google_pay
4 → atm_support
5 → automatic_top_up
6 → balance_not_updated_after_bank_transfer
7 → balance_not_updated_after_cheque_or_cash_deposit
8 → beneficiary_not_allowed
9 → cancel_transfer


Create a validation set


In [7]:
from sklearn.model_selection import train_test_split

train_split, val_split = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

print("Training samples:", len(train_split))
print("Validation samples:", len(val_split))
print("Official test samples:", len(test_df))

Training samples: 8502
Validation samples: 1501
Official test samples: 3080


Convert to Hugging Face datasets

In [8]:
train_dataset = Dataset.from_pandas(
    train_split[["text", "label"]],
    preserve_index=False
)

val_dataset = Dataset.from_pandas(
    val_split[["text", "label"]],
    preserve_index=False
)

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

Dataset({
    features: ['text', 'label'],
    num_rows: 8502
})
Dataset({
    features: ['text', 'label'],
    num_rows: 1501
})
Dataset({
    features: ['text', 'label'],
    num_rows: 3080
})


Load DistilBERT tokenizer

In [9]:
MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("Tokenizer loaded successfully")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

c:\Users\GOPINATH\ALML\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GOPINATH\.cache\huggingface\hub\models--distilbert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Tokenizer loaded successfully


In [10]:
MAX_LENGTH = 128

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

In [11]:
train_tokenized = train_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

Map:   0%|          | 0/8502 [00:00<?, ? examples/s]

Map:   0%|          | 0/1501 [00:00<?, ? examples/s]

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

11. Load DistilBERT classification model

In [12]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(categories),
    id2label=id2label,
    label2id=label2id
)

print("DistilBERT model loaded.")
print("Number of classes:", len(categories))

c:\Users\GOPINATH\ALML\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\GOPINATH\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


DistilBERT model loaded.
Number of classes: 77


12. Define evaluation metrics

In [13]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

13. Configure training

In [15]:
training_args = TrainingArguments(
    output_dir="../Models/distilbert_banking",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=100,
    report_to="none"
)

print("TrainingArguments created successfully!")

TrainingArguments created successfully!


In [16]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer created successfully!")

Trainer created successfully!


15. Start fine-tuning

In [17]:
trainer.train()

c:\Users\GOPINATH\ALML\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,1.890196,1.537525,0.733511,0.748809,0.733511,0.705745
2,0.846697,0.728351,0.874084,0.879605,0.874084,0.871700
3,0.586734,0.580889,0.879414,0.883322,0.879414,0.877125


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\GOPINATH\ALML\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\GOPINATH\ALML\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3189, training_loss=1.5296331939745114, metrics={'train_runtime': 50652.7681, 'train_samples_per_second': 0.504, 'train_steps_per_second': 0.063, 'total_flos': 845808140496384.0, 'train_loss': 1.5296331939745114, 'epoch': 3.0})

In [2]:
import os

MODEL_DIR = "../Models/distilbert_banking"

print("Model folder exists:", os.path.exists(MODEL_DIR))

if os.path.exists(MODEL_DIR):
    print("\nFiles:")
    print(os.listdir(MODEL_DIR))

Model folder exists: True

Files:
['checkpoint-1063', 'checkpoint-2126', 'checkpoint-3189']


In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_DIR = "../Models/distilbert_banking/checkpoint-3189"

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_DIR
)

print("DistilBERT checkpoint loaded successfully!")
print("Number of labels:", model.config.num_labels)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

DistilBERT checkpoint loaded successfully!
Number of labels: 77


In [4]:
import pandas as pd

test_df = pd.read_csv("../data/test.csv")

print("Test shape:", test_df.shape)
print("Columns:", test_df.columns.tolist())

Test shape: (3080, 2)
Columns: ['text', 'category']


In [5]:
train_df = pd.read_csv("../data/train.csv")

categories = sorted(train_df["category"].unique())

label2id = {
    label: idx
    for idx, label in enumerate(categories)
}

id2label = {
    idx: label
    for label, idx in label2id.items()
}

test_df["label"] = test_df["category"].map(label2id)

print("Number of categories:", len(categories))
print("Test labels created:", test_df["label"].notna().sum())

Number of categories: 77
Test labels created: 3080


In [6]:
from datasets import Dataset

MAX_LENGTH = 128

test_dataset = Dataset.from_pandas(
    test_df[["text", "label"]],
    preserve_index=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH
    )

test_tokenized = test_dataset.map(
    tokenize_function,
    batched=True
)

print("Test dataset tokenized successfully!")

Map:   0%|          | 0/3080 [00:00<?, ? examples/s]

Test dataset tokenized successfully!


In [7]:
from transformers import Trainer
import numpy as np

trainer = Trainer(
    model=model,
    processing_class=tokenizer
)

print("Trainer ready for evaluation!")

Trainer ready for evaluation!


In [8]:
test_output = trainer.predict(test_tokenized)

logits = test_output.predictions
true_labels = test_output.label_ids

predicted_labels = np.argmax(logits, axis=-1)

print("FINAL TEST EVALUATION")
print("---------------------")
print("Test samples:", len(true_labels))
print("Predictions:", len(predicted_labels))

c:\Users\GOPINATH\ALML\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


FINAL TEST EVALUATION
---------------------
Test samples: 3080
Predictions: 3080


In [9]:
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

accuracy = accuracy_score(
    true_labels,
    predicted_labels
)

precision, recall, f1, _ = precision_recall_fscore_support(
    true_labels,
    predicted_labels,
    average="weighted",
    zero_division=0
)

print("DistilBERT FINAL TEST RESULTS")
print("-----------------------------")
print(f"Accuracy : {accuracy:.4f} ({accuracy * 100:.2f}%)")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

DistilBERT FINAL TEST RESULTS
-----------------------------
Accuracy : 0.8708 (87.08%)
Precision: 0.8699
Recall   : 0.8708
F1 Score : 0.8657


The fine-tuned DistilBERT model achieved 87.08% test accuracy and a weighted F1-score of 86.57%. The Balanced SVM baseline achieved 89.64% accuracy and weighted F1-score of 89.64% on the same test dataset.

Convert logits to probabilities - Phase 3

In [10]:
import torch
import numpy as np

probabilities = torch.softmax(
    torch.tensor(logits),
    dim=-1
).numpy()

print("Probability shape:", probabilities.shape)

Probability shape: (3080, 77)


In [11]:
predicted_labels = np.argmax(probabilities, axis=1)

predicted_categories = [
    id2label[label]
    for label in predicted_labels
]

confidence_scores = np.max(
    probabilities,
    axis=1
)

print("First 10 predictions:")

for i in range(10):
    print(
        test_df.iloc[i]["text"],
        "→",
        predicted_categories[i],
        f"({confidence_scores[i] * 100:.2f}%)"
    )

First 10 predictions:
How do I locate my card? → card_linking (32.78%)
I still have not received my new card, I ordered over a week ago. → card_arrival (84.99%)
I ordered a card but it has not arrived. Help please! → card_arrival (77.66%)
Is there a way to know when my card will arrive? → card_delivery_estimate (47.96%)
My card has not arrived yet. → card_arrival (82.04%)
When will I get my card? → card_delivery_estimate (72.52%)
Do you know if there is a tracking number for the new card you sent me? → card_arrival (84.77%)
i have not received my card → card_arrival (77.17%)
still waiting on that card → card_arrival (83.38%)
Is it normal to have to wait over a week for my new card? → card_arrival (79.48%)


In [12]:
top_3_indices = np.argsort(
    probabilities,
    axis=1
)[:, -3:][:, ::-1]

top_3_results = []

for i in range(len(test_df)):

    top_3 = []

    for idx in top_3_indices[i]:
        top_3.append({
            "category": id2label[idx],
            "probability": float(probabilities[i][idx])
        })

    top_3_results.append(top_3)

In [15]:
print("trainer:", "trainer" in globals())
print("model:", "model" in globals())
print("tokenizer:", "tokenizer" in globals())
print("train_df:", "train_df" in globals())

trainer: True
model: True
tokenizer: True
train_df: True


In [16]:
from sklearn.model_selection import train_test_split
from datasets import Dataset

# Create numeric labels
train_df["label"] = train_df["category"].map(label2id)

# Recreate the SAME 85/15 split
train_part, val_part = train_test_split(
    train_df,
    test_size=0.15,
    random_state=42,
    stratify=train_df["label"]
)

print("Training samples:", len(train_part))
print("Validation samples:", len(val_part))

Training samples: 8502
Validation samples: 1501


In [17]:
val_dataset = Dataset.from_pandas(
    val_part[["text", "label"]],
    preserve_index=False
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

val_tokenized = val_dataset.map(
    tokenize_function,
    batched=True
)

val_tokenized = val_tokenized.remove_columns(["text"])

print(val_tokenized)

Map:   0%|          | 0/1501 [00:00<?, ? examples/s]

Dataset({
    features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 1501
})


In [18]:
val_output = trainer.predict(val_tokenized)

val_logits = val_output.predictions
val_true_labels = val_output.label_ids

val_probabilities = torch.softmax(
    torch.tensor(val_logits),
    dim=-1
).numpy()

val_predicted_labels = np.argmax(
    val_probabilities,
    axis=1
)

val_confidence = np.max(
    val_probabilities,
    axis=1
)

val_correct = (
    val_predicted_labels == val_true_labels
)

print("Validation samples:", len(val_true_labels))
print("Correct:", val_correct.sum())
print("Incorrect:", (~val_correct).sum())

c:\Users\GOPINATH\ALML\.venv\Lib\site-packages\torch\utils\data\dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Validation samples: 1501
Correct: 1320
Incorrect: 181


In [19]:
print(
    "Average confidence - correct:",
    round(val_confidence[val_correct].mean() * 100, 2),
    "%"
)

print(
    "Average confidence - incorrect:",
    round(val_confidence[~val_correct].mean() * 100, 2),
    "%"
)

Average confidence - correct: 74.81 %
Average confidence - incorrect: 41.76 %


In [20]:
thresholds = [
    0.50,
    0.55,
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90
]

threshold_results = []

for threshold in thresholds:

    selected = val_confidence >= threshold

    if selected.sum() == 0:
        continue

    accuracy = val_correct[selected].mean()
    coverage = selected.mean()

    threshold_results.append({
        "Threshold": threshold,
        "Accepted Accuracy": accuracy,
        "Coverage": coverage,
        "Escalation Rate": 1 - coverage,
        "Samples": selected.sum()
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df["Accepted Accuracy (%)"] = (
    threshold_df["Accepted Accuracy"] * 100
)

threshold_df["Coverage (%)"] = (
    threshold_df["Coverage"] * 100
)

threshold_df["Escalation Rate (%)"] = (
    threshold_df["Escalation Rate"] * 100
)

threshold_df[
    [
        "Threshold",
        "Accepted Accuracy (%)",
        "Coverage (%)",
        "Escalation Rate (%)",
        "Samples"
    ]
].round(2)

,Threshold,Accepted Accuracy (%),Coverage (%),Escalation Rate (%),Samples
0,0.50,95.43,83.01,16.99,1246
1,0.55,96.34,78.28,21.72,1175
2,0.60,97.38,73.82,26.18,1108
3,0.65,98.27,69.15,30.85,1038
4,0.70,98.74,63.62,36.38,955
5,0.75,99.16,55.36,44.64,831
6,0.80,99.39,43.97,56.03,660
7,0.85,100.00,31.11,68.89,467
8,0.90,100.00,12.66,87.34,190


Phase -4

HIGH CONFIDENCE
≥ 75%
→ Automated response

MEDIUM CONFIDENCE
50% – 74.99%
→ Response + clarification / additional verification

LOW CONFIDENCE
< 50%
→ Human-agent escalation

Implement the confidence decision layer

In [21]:
def confidence_decision(confidence):
    
    if confidence >= 0.75:
        return "HIGH", "Automated response"
    
    elif confidence >= 0.50:
        return "MEDIUM", "Clarification required"
    
    else:
        return "LOW", "Human-agent escalation"

In [22]:
test_confidences = [
    0.90,
    0.78,
    0.65,
    0.52,
    0.32
]

for confidence in test_confidences:
    level, action = confidence_decision(confidence)
    
    print(
        f"{confidence*100:.2f}% "
        f"→ {level} → {action}"
    )

90.00% → HIGH → Automated response
78.00% → HIGH → Automated response
65.00% → MEDIUM → Clarification required
52.00% → MEDIUM → Clarification required
32.00% → LOW → Human-agent escalation


In [23]:
confidence = 0.3278

level, action = confidence_decision(confidence)

print("Confidence:", f"{confidence*100:.2f}%")
print("Decision:", level)
print("Action:", action)

Confidence: 32.78%
Decision: LOW
Action: Human-agent escalation


Customer Query
      ↓
DistilBERT
      ↓
Predicted Intent
      ↓
Confidence Score
      ↓
┌──────────────────────────┐
│ Confidence Decision Layer│
└──────────────────────────┘
      ↓
 ┌────────┬──────────┬─────────┐
 HIGH     MEDIUM       LOW
 ≥75%     50–74.99%   <50%
   ↓          ↓          ↓
Auto       Verify      Human
response   /clarify    agent

Confidence & Escalation Layer - Phase 4

In [24]:
query = "How do I locate my card?"

predicted_category = "card_linking"
confidence = 0.3278

level, action = confidence_decision(confidence)

print("Customer query:", query)
print("Predicted category:", predicted_category)
print("Confidence:", f"{confidence*100:.2f}%")
print("Confidence level:", level)
print("Recommended action:", action)

Customer query: How do I locate my card?
Predicted category: card_linking
Confidence: 32.78%
Confidence level: LOW
Recommended action: Human-agent escalation


Add a complete prediction function

Now we'll combine DistilBERT prediction + confidence + top 3 predictions + escalation decision.

In [25]:
def predict_with_confidence(query, model, tokenizer, id2label):
    
    # Tokenize input
    inputs = tokenizer(
        query,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )
    
    # Prediction
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Convert logits to probabilities
    probabilities = torch.softmax(
        outputs.logits,
        dim=-1
    )[0]
    
    # Top 3 predictions
    top_values, top_indices = torch.topk(
        probabilities,
        k=3
    )
    
    top_predictions = []
    
    for probability, index in zip(
        top_values,
        top_indices
    ):
        top_predictions.append({
            "category": id2label[int(index)],
            "confidence": float(probability)
        })
    
    # Best prediction
    predicted_category = top_predictions[0]["category"]
    confidence = top_predictions[0]["confidence"]
    
    # Confidence decision
    level, action = confidence_decision(confidence)
    
    return {
        "query": query,
        "predicted_category": predicted_category,
        "confidence": confidence,
        "confidence_level": level,
        "action": action,
        "top_predictions": top_predictions
    }

In [27]:
result = predict_with_confidence(
    "How do I locate my card?",
    model,
    tokenizer,
    id2label
)

print("Query:", result["query"])
print("Predicted:", result["predicted_category"])
print(
    "Confidence:",
    f"{result['confidence']*100:.2f}%"
)
print("Level:", result["confidence_level"])
print("Action:", result["action"])

print("\nTop 3 predictions:")

for prediction in result["top_predictions"]:
    print(
        prediction["category"],
        f"{prediction['confidence']*100:.2f}%"
    )

Query: How do I locate my card?
Predicted: card_linking
Confidence: 32.78%
Level: LOW
Action: Human-agent escalation

Top 3 predictions:
card_linking 32.78%
card_arrival 25.83%
lost_or_stolen_card 11.97%


Intent:
card_linking

Confidence:
32.78%

Confidence level:
LOW

Action:
Human-agent escalation

Top alternatives:
1. card_linking       32.78%
2. card_arrival       25.83%
3. lost_or_stolen_card 11.97%

Phase 5 — Generative AI Customer Response

Step 5.1 — Create controlled support context

In [28]:
support_context = {
    
    "card_arrival": {
        "description": "The customer is asking about the arrival or delivery status of their physical card.",
        "recommended_action": "Provide general card-delivery guidance and ask for tracking information if needed."
    },
    
    "card_delivery_estimate": {
        "description": "The customer wants to know approximately when their card will arrive.",
        "recommended_action": "Provide general delivery-time guidance if available and avoid promising an exact delivery date."
    },
    
    "card_linking": {
        "description": "The customer is asking about linking or connecting an existing card to their banking account or application.",
        "recommended_action": "Provide general card-linking guidance and ask for clarification if the customer's request is ambiguous."
    },
    
    "lost_or_stolen_card": {
        "description": "The customer reports that their card may be lost or stolen.",
        "recommended_action": "Recommend using the bank's official card-security process and contacting the bank if the card is lost or stolen."
    },
    
    "card_not_working": {
        "description": "The customer's card is not working when they attempt to use it.",
        "recommended_action": "Provide general troubleshooting guidance and recommend contacting the bank if the issue continues."
    },
    
    "top_up_failed": {
        "description": "The customer reports that a card or account top-up failed.",
        "recommended_action": "Explain general troubleshooting steps and recommend checking the payment method or contacting support."
    }
}

print("Support contexts:", len(support_context))

Support contexts: 6


Step 5.2 — Create the GenAI prompt

In [29]:
def create_support_prompt(
    query,
    predicted_category,
    confidence,
    confidence_level,
    top_predictions
):
    
    context = support_context.get(
        predicted_category,
        {
            "description": "No verified support context is available for this intent.",
            "recommended_action": "Ask the customer for clarification or escalate to a human agent."
        }
    )
    
    top_prediction_text = "\n".join(
        [
            f"- {item['category']}: "
            f"{item['confidence']*100:.2f}%"
            for item in top_predictions
        ]
    )
    
    prompt = f"""
You are a professional banking customer-support assistant.

Your task is to respond to the customer's message using ONLY
the verified information supplied below.

CUSTOMER MESSAGE:
{query}

PREDICTED INTENT:
{predicted_category}

MODEL CONFIDENCE:
{confidence*100:.2f}%

CONFIDENCE LEVEL:
{confidence_level}

TOP PREDICTIONS:
{top_prediction_text}

VERIFIED SUPPORT CONTEXT:
{context['description']}

RECOMMENDED ACTION:
{context['recommended_action']}

RULES:
1. Do not invent account information.
2. Do not invent transaction details.
3. Do not claim that an action has been completed.
4. Do not invent bank policies, fees, dates, or processing times.
5. Do not request passwords, PINs, OTPs, or full card numbers.
6. If confidence is LOW, do not assume that the predicted intent is correct.
7. For LOW confidence, politely ask the customer to clarify their request
   or recommend human-agent assistance.
8. Keep the response concise, professional, and helpful.
9. Never reveal internal model scores or classification details to the customer.

Generate the customer-facing response.
"""
    
    return prompt

Step 5.3 — Test the prompt with your actual example

In [30]:
result = predict_with_confidence(
    "How do I locate my card?",
    model,
    tokenizer,
    id2label
)

prompt = create_support_prompt(
    query=result["query"],
    predicted_category=result["predicted_category"],
    confidence=result["confidence"],
    confidence_level=result["confidence_level"],
    top_predictions=result["top_predictions"]
)

print(prompt)


You are a professional banking customer-support assistant.

Your task is to respond to the customer's message using ONLY
the verified information supplied below.

CUSTOMER MESSAGE:
How do I locate my card?

PREDICTED INTENT:
card_linking

MODEL CONFIDENCE:
32.78%

CONFIDENCE LEVEL:
LOW

TOP PREDICTIONS:
- card_linking: 32.78%
- card_arrival: 25.83%
- lost_or_stolen_card: 11.97%

VERIFIED SUPPORT CONTEXT:
The customer is asking about linking or connecting an existing card to their banking account or application.

RECOMMENDED ACTION:
Provide general card-linking guidance and ask for clarification if the customer's request is ambiguous.

RULES:
1. Do not invent account information.
2. Do not invent transaction details.
3. Do not claim that an action has been completed.
4. Do not invent bank policies, fees, dates, or processing times.
5. Do not request passwords, PINs, OTPs, or full card numbers.
6. If confidence is LOW, do not assume that the predicted intent is correct.
7. For LOW confi

Step 5.5 — Add a safe response policy

In [31]:
def response_policy(confidence_level):
    
    if confidence_level == "HIGH":
        return (
            "Answer the customer's question using the verified support context."
        )
    
    elif confidence_level == "MEDIUM":
        return (
            "Do not assume the predicted intent is completely correct. "
            "Provide limited guidance and ask a clarification question."
        )
    
    else:
        return (
            "Do not provide an intent-specific answer. "
            "Ask the customer to clarify what they mean or recommend "
            "human-agent assistance."
        )

In [32]:
for level in ["HIGH", "MEDIUM", "LOW"]:
    print(level, "→", response_policy(level))

HIGH → Answer the customer's question using the verified support context.
MEDIUM → Do not assume the predicted intent is completely correct. Provide limited guidance and ask a clarification question.
LOW → Do not provide an intent-specific answer. Ask the customer to clarify what they mean or recommend human-agent assistance.


Step 5.1.5 — Load the API key in your notebook

In [37]:

from pathlib import Path

print("Current working directory:")
print(Path.cwd())

print("\n.env locations:")

for path in [
    Path.cwd() / ".env",
    Path.cwd().parent / ".env"
]:
    print(path, "→", path.exists())

Current working directory:
c:\Users\GOPINATH\ALML\notebooks

.env locations:
c:\Users\GOPINATH\ALML\notebooks\.env → False
c:\Users\GOPINATH\ALML\.env → True


In [38]:
from pathlib import Path
from dotenv import load_dotenv
import os

project_root = Path.cwd().parent
env_path = project_root / ".env"

print("Looking for .env at:")
print(env_path)

print("Exists:", env_path.exists())

load_dotenv(env_path)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError(
        "GEMINI_API_KEY was not found in the project .env file."
    )

print("Gemini API key loaded successfully.")

Looking for .env at:
c:\Users\GOPINATH\ALML\.env
Exists: True


ValueError: GEMINI_API_KEY was not found in the project .env file.

In [43]:
from pathlib import Path

env_path = Path(r"C:\Users\GOPINATH\ALML\.env")

print("Exists:", env_path.exists())
print("Is file:", env_path.is_file())
print("Is directory:", env_path.is_dir())

Exists: False
Is file: False
Is directory: False


In [48]:
from pathlib import Path

env_path = Path(r"C:\Users\GOPINATH\ALML\.env")

print("Exists:", env_path.exists())
print("Is file:", env_path.is_file())

Exists: True
Is file: True


In [49]:
content = env_path.read_text(encoding="utf-8")

for line in content.splitlines():
    line = line.strip()
    if line and not line.startswith("#"):
        print("Variable found:", line.split("=", 1)[0].strip())

Variable found: GEMINI_API_KEY


In [50]:
from dotenv import load_dotenv
import os

env_path = r"C:\Users\GOPINATH\ALML\.env"

load_dotenv(env_path, override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

print("Key found:", GEMINI_API_KEY is not None)
print("Key length:", len(GEMINI_API_KEY) if GEMINI_API_KEY else 0)

Key found: True
Key length: 53


Create the Gemini client

In [1]:
from google import genai
import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY was not found.")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client created successfully")

Gemini client created successfully


In [4]:
GEMINI_MODEL = "gemini-3.5-flash-lite"

response = client.models.generate_content(
    model=GEMINI_MODEL,
    contents="Say hello and confirm that you are ready for a banking customer-support project."
)

print(response.text)

Hello! I am ready for the banking customer-support project. Please let me know the details, guidelines, or the first task whenever you are ready.


Phase 5.2 — Build the controlled GenAI support function

In [6]:
from pydantic import BaseModel, Field


class SupportResponse(BaseModel):
    customer_response: str
    customer_issue: str
    summary: str
    sentiment: str
    priority: str
    recommended_action: str
    escalation_required: bool

Controlled prompt

In [7]:
def generate_support_response(
    query,
    predicted_category,
    confidence,
    confidence_level,
    top_predictions,
    support_context
):

    top_predictions_text = "\n".join(
        [
            f"- {category}: {probability:.2%}"
            for category, probability in top_predictions
        ]
    )

    prompt = f"""
You are a professional banking customer-support assistant.

Your job is to assist a customer using ONLY the verified support
information supplied below.

CUSTOMER MESSAGE:
{query}

PREDICTED INTENT:
{predicted_category}

MODEL CONFIDENCE:
{confidence:.2%}

CONFIDENCE LEVEL:
{confidence_level}

TOP PREDICTIONS:
{top_predictions_text}

VERIFIED SUPPORT CONTEXT:
{support_context}

STRICT RULES:

1. Use only the verified support context.
2. Do not invent banking policies, fees, limits, transaction details,
   delivery dates, or account information.
3. Never claim that an action has been completed.
4. Never ask for passwords, PINs, OTPs, CVV, or full card numbers.
5. Never reveal model confidence or internal classification details
   in the customer-facing response.
6. If confidence is LOW, do not assume that the predicted intent
   is definitely correct.
7. If confidence is LOW, recommend clarification or human-agent review.
8. If confidence is MEDIUM, provide cautious guidance and request
   clarification when necessary.
9. If confidence is HIGH, provide a concise response using the
   verified context.
10. Keep the customer response professional and concise.
11. The recommended action is for the support team, not the customer.
12. Identify customer sentiment and priority from the message only.
13. Do not invent information that is not present in the context.

Return the result using the required structured format.
"""

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": SupportResponse
        }
    )

    return SupportResponse.model_validate_json(response.text)

Phase 5.4 — Use your existing example

In [11]:
query = "I forgot my password"

predicted_category = "card_linking"

confidence = 0.3278

confidence_level = "LOW"

top_predictions = [
    ("card_linking", 0.3278),
    ("card_arrival", 0.2583),
    ("lost_or_stolen_card", 0.1197)
]

support_context = """
The customer may be asking about connecting or linking an existing
card to their banking account or application.

If the customer's meaning is unclear, ask for clarification rather
than assuming the intent.
"""

In [12]:
result = generate_support_response(
    query=query,
    predicted_category=predicted_category,
    confidence=confidence,
    confidence_level=confidence_level,
    top_predictions=top_predictions,
    support_context=support_context
)

print("CUSTOMER RESPONSE:")
print(result.customer_response)

print("\nCUSTOMER ISSUE:")
print(result.customer_issue)

print("\nSUMMARY:")
print(result.summary)

print("\nSENTIMENT:")
print(result.sentiment)

print("\nPRIORITY:")
print(result.priority)

print("\nRECOMMENDED ACTION:")
print(result.recommended_action)

print("\nESCALATION REQUIRED:")
print(result.escalation_required)

CUSTOMER RESPONSE:
Could you please clarify whether you are asking about connecting or linking an existing card to your banking account or application?

CUSTOMER ISSUE:
Forgot password

SUMMARY:
The customer stated they forgot their password, but confidence for the predicted intent is low and clarification is needed.

SENTIMENT:
neutral

PRIORITY:
medium

RECOMMENDED ACTION:
Review the customer message and ask for clarification or route to a human agent.

ESCALATION REQUIRED:
True


Phase 5.6 — Fix hallucination / context contamination

1. Restart the test with fresh variables

In [2]:
test_query = "How do I locate my card?"

test_predicted_category = "card_linking"

test_confidence = 0.3278

test_confidence_level = "LOW"

test_top_predictions = [
    ("card_linking", 0.3278),
    ("card_arrival", 0.2583),
    ("lost_or_stolen_card", 0.1197)
]

test_support_context = """
Verified information:

The customer may be asking about connecting or linking
an existing card to their banking account or application.

If the customer's request is ambiguous, clarification
should be requested.

No additional customer information is available.
"""

In [3]:
def generate_support_response(
    query,
    predicted_category,
    confidence,
    confidence_level,
    top_predictions,
    support_context
):

    top_predictions_text = "\n".join(
        f"- {category}: {probability:.2%}"
        for category, probability in top_predictions
    )

    prompt = f"""
You are a professional banking customer-support assistant.

IMPORTANT:
You must treat the CUSTOMER MESSAGE below as the ONLY source
for identifying the customer's actual issue.

CUSTOMER MESSAGE:
<<<
{query}
>>>

PREDICTED INTENT:
{predicted_category}

MODEL CONFIDENCE:
{confidence:.2%}

CONFIDENCE LEVEL:
{confidence_level}

TOP PREDICTIONS:
{top_predictions_text}

VERIFIED SUPPORT CONTEXT:
<<<
{support_context}
>>>

STRICT RULES:

1. Never invent information about the customer.
2. Never use information from previous conversations, previous
   examples, previous requests, or previous Python variables.
3. The customer issue must be based ONLY on the current
   CUSTOMER MESSAGE.
4. Do not claim the customer has forgotten a password unless
   the current CUSTOMER MESSAGE explicitly says so.
5. Do not claim the customer has lost a card unless the current
   CUSTOMER MESSAGE explicitly says so.
6. Do not invent account, transaction, card, or personal details.
7. Use only the VERIFIED SUPPORT CONTEXT for support guidance.
8. If confidence is LOW, treat the predicted intent as uncertain.
9. For LOW confidence, ask a clarification question rather than
   assuming the prediction is correct.
10. Never expose model confidence, probabilities, or internal
    classification details to the customer.
11. Never request passwords, PINs, OTPs, CVVs, or full card numbers.
12. Do not claim that an action has been completed.
13. The customer issue must be a short factual description of
    what the customer actually said.
14. Sentiment must be inferred only from the current customer message.
15. Priority must be based only on the current customer message.
16. If there is insufficient information, say so instead of guessing.

Return:
- customer_response
- customer_issue
- summary
- sentiment
- priority
- recommended_action
- escalation_required
"""

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": SupportResponse
        }
    )

    return SupportResponse.model_validate_json(response.text)

In [4]:
result = generate_support_response(
    query=test_query,
    predicted_category=test_predicted_category,
    confidence=test_confidence,
    confidence_level=test_confidence_level,
    top_predictions=test_top_predictions,
    support_context=test_support_context
)

print("CUSTOMER RESPONSE:")
print(result.customer_response)

print("\nCUSTOMER ISSUE:")
print(result.customer_issue)

print("\nSUMMARY:")
print(result.summary)

print("\nSENTIMENT:")
print(result.sentiment)

print("\nPRIORITY:")
print(result.priority)

print("\nRECOMMENDED ACTION:")
print(result.recommended_action)

print("\nESCALATION REQUIRED:")
print(result.escalation_required)

NameError: name 'client' is not defined

Phase 6 — Conversation Intelligence

Customer Query
      ↓
DistilBERT
      ↓
Intent + Confidence + Top 3
      ↓
Confidence & Escalation
      ↓
Gemini
      ↓
┌─────────────────────────────┐
│ Conversation Intelligence   │
├─────────────────────────────┤
│ Customer Issue              │
│ Intent                      │
│ Sentiment                   │
│ Priority                    │
│ Key Information             │
│ Recommended Action          │
│ Customer Response           │
│ Escalation                  │
└─────────────────────────────┘

In [16]:
class SupportResponse(BaseModel):
    customer_response: str
    customer_issue: str
    predicted_intent: str
    sentiment: str
    priority: str
    key_information: list[str]
    recommended_action: str
    escalation_required: bool
    summary: str

Phase 6.1 — Update the output schema

In [17]:
from pydantic import BaseModel, Field


class ConversationIntelligence(BaseModel):
    customer_response: str
    customer_issue: str
    predicted_intent: str
    sentiment: str
    priority: str
    key_information: list[str]
    recommended_action: str
    escalation_required: bool
    summary: str

print("Conversation Intelligence schema created successfully")

Conversation Intelligence schema created successfully


Phase 6.2 — Create the conversation-intelligence function

In [22]:
def analyze_conversation(
    query,
    predicted_category,
    confidence,
    confidence_level,
    top_predictions,
    support_context
):

    top_predictions_text = "\n".join(
        f"- {category}: {probability:.2%}"
        for category, probability in top_predictions
    )

    prompt = f"""
You are a professional banking customer-support
conversation intelligence assistant.

CURRENT CUSTOMER MESSAGE:
<<<
{query}
>>>

ML PREDICTED INTENT:
{predicted_category}

MODEL CONFIDENCE:
{confidence:.2%}

CONFIDENCE LEVEL:
{confidence_level}

TOP PREDICTIONS:
{top_predictions_text}

VERIFIED SUPPORT CONTEXT:
<<<
{support_context}
>>>

Analyze ONLY the current customer message.

RULES:

1. Never use information from previous conversations or examples.
2. Never invent customer information.
3. Never invent account, transaction, card, or personal details.
4. Do not assume the predicted intent is correct when confidence is LOW.
5. If the request is ambiguous, clearly identify the ambiguity.
6. Extract key information ONLY from the current customer message.
7. If useful information exists, provide it as short bullet-style items.
8. If there is very little information, include a short factual item
   such as "Customer is asking about locating a card."
9. Never fabricate missing information.
10. Sentiment must be based only on the current message.
11. Priority must be based only on the current message.
12. Never expose internal model probabilities to the customer.
13. Never request passwords, PINs, OTPs, CVVs, or full card numbers.
14. Never claim an action has been completed.
15. LOW confidence MUST result in escalation_required = true.
16. MEDIUM confidence may require escalation if clarification is needed.
17. HIGH confidence can normally proceed without escalation.
18. Customer response must be concise and professional.

The customer_response must be suitable for directly showing to the customer.

The recommended_action is for the support team.

Return:
customer_response
customer_issue
predicted_intent
sentiment
priority
key_information
recommended_action
escalation_required
summary
"""

    response = client.models.generate_content(
        model=GEMINI_MODEL,
        contents=prompt,
        config={
            "response_mime_type": "application/json",
            "response_schema": ConversationIntelligence
        }
    )

    return ConversationIntelligence.model_validate_json(response.text)

In [19]:
test_query = "How do I locate my card?"

test_predicted_category = "card_linking"

test_confidence = 0.3278

test_confidence_level = "LOW"

test_top_predictions = [
    ("card_linking", 0.3278),
    ("card_arrival", 0.2583),
    ("lost_or_stolen_card", 0.1197)
]

test_support_context = """
Verified support information:

The customer may be asking about connecting or linking
an existing card to their banking account or application.

The request may also require clarification if the customer
is referring to card delivery or another card-related issue.

No additional customer information is available.
"""

In [23]:
conversation_result = analyze_conversation(
    query=test_query,
    predicted_category=test_predicted_category,
    confidence=test_confidence,
    confidence_level=test_confidence_level,
    top_predictions=test_top_predictions,
    support_context=test_support_context
)

Phase 6.4 — Display the results

In [1]:
print("========== CONVERSATION INTELLIGENCE ==========")

print("\nCUSTOMER RESPONSE:")
print(conversation_result.customer_response)

print("\nCUSTOMER ISSUE:")
print(conversation_result.customer_issue)

print("\nPREDICTED INTENT:")
print(conversation_result.predicted_intent)

print("\nSENTIMENT:")
print(conversation_result.sentiment)

print("\nPRIORITY:")
print(conversation_result.priority)

print("\nKEY INFORMATION:")
for item in conversation_result.key_information:
    print("-", item)

print("\nRECOMMENDED ACTION:")
print(conversation_result.recommended_action)

print("\nESCALATION REQUIRED:")
print(conversation_result.escalation_required)

print("\nSUMMARY:")
print(conversation_result.summary)

========== CONVERSATION INTELLIGENCE ==========

CUSTOMER RESPONSE:


NameError: name 'conversation_result' is not defined

In [28]:
from pathlib import Path

print(Path(MODEL_PATH).resolve())
print(Path(MODEL_PATH).exists())

C:\Users\GOPINATH\ALML\Models\distilbert_banking\checkpoint-3189
True


In [29]:
import pandas as pd

test_df = pd.read_csv("../data/test.csv")

test_df[["text", "category"]].sample(10, random_state=42)

,text,category
63,How do I link this new card?,card_linking
1921,How do I retrieve my card from the machine?,card_swallowed
2018,I want to know where the funds come from.,verify_source_of_funds
331,"I just activated auto top-up, but it is not le...",automatic_top_up
810,Why did I have to pay extra because I paid wit...,card_payment_fee_charged
594,How can I fix a problem where contactless isn'...,contactless_not_working
2580,I took out a foreign currency and the exchange...,wrong_exchange_rate_for_cash_withdrawal
346,Will it automatically top-up money if there is...,automatic_top_up
1029,"I see my top-up was canceled, but why?",top_up_reverted
1675,I lost my phone!,lost_or_stolen_phone


In [30]:
print(GEMINI_MODEL)

gemini-3.5-flash-lite
